In [1]:
import xarray as xr
import numpy as np
import json
import pandas as pd
import os
import glob
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from global_land_mask import globe 
import warnings

# Matikan warning agar output bersih
warnings.filterwarnings("ignore")

def process_wind_images_final_v3():
    print("GENERATOR V4: GRID WIND ROSES (DYNAMIC CLIM/SEAS)")

    # ==========================================
    # 0. KONFIGURASI VISUAL
    # ==========================================
    CMAP_MAP = "inferno"     
    ARROW_COLOR = "#FFFFFF"  
    COASTLINE_COLOR = "#000000" 
    TEXT_COLOR = "#e0e0e0"
    
    DPI_MAP = 150
    DPI_ROSE = 100 
    
    # 1. KONFIGURASI DATA
    start_year = 1979
    end_year = 2024
    data_folder = '.' 
    
    # Folder Output
    base_dir = "frames_wind_final"
    dirs = {
        "map_ts": f"{base_dir}/map/ts",
        "map_clim": f"{base_dir}/map/clim",
        "map_seas": f"{base_dir}/map/seas",
        "rose_area_ts": f"{base_dir}/rose/area/ts",
        "rose_area_clim": f"{base_dir}/rose/area/clim",
        "rose_area_seas": f"{base_dir}/rose/area/seas",
        # [MODIFIKASI] Folder Grid dipisah
        "rose_grid_ts": f"{base_dir}/rose/grid/ts",     # Static Average
        "rose_grid_clim": f"{base_dir}/rose/grid/clim", # Dynamic per Month
        "rose_grid_seas": f"{base_dir}/rose/grid/seas"  # Dynamic per Season
    }
    for d in dirs.values():
        os.makedirs(d, exist_ok=True)

    monthly_ds_u = []
    monthly_ds_v = []
    monthly_ds_speed = []

    # ==========================================
    # 2. LOAD DATA
    # ==========================================
    print(f"Membaca Data Angin (U & V) {start_year}-{end_year}...")

    for year in range(start_year, end_year + 1):
        u_files = glob.glob(os.path.join(data_folder, f"*u_component*{year}*.nc"))
        v_files = glob.glob(os.path.join(data_folder, f"*v_component*{year}*.nc"))
        
        if not u_files or not v_files: continue
        
        try:
            ds_u = xr.open_dataset(u_files[0])
            ds_v = xr.open_dataset(v_files[0])
            
            if 'valid_time' in ds_u.coords: ds_u = ds_u.rename({'valid_time': 'time'})
            if 'valid_time' in ds_v.coords: ds_v = ds_v.rename({'valid_time': 'time'})
            
            var_u = next((v for v in ['u10', 'u', 'var165'] if v in ds_u), None)
            var_v = next((v for v in ['v10', 'v', 'var166'] if v in ds_v), None)
            
            if var_u and var_v:
                u_m = ds_u[var_u].resample(time='1M').mean()
                v_m = ds_v[var_v].resample(time='1M').mean()
                speed_m = np.sqrt(u_m**2 + v_m**2)
                
                monthly_ds_u.append(u_m)
                monthly_ds_v.append(v_m)
                monthly_ds_speed.append(speed_m)
                
            ds_u.close(); ds_v.close()
            print(f"   [{year}] OK", end='\r')
        except Exception as e: 
            print(f"Error {year}: {e}")

    if not monthly_ds_u:
        print("\nGagal loading data.")
        return

    print("\nMenggabungkan Dataset...")
    ds_u = xr.concat(monthly_ds_u, dim='time')
    ds_v = xr.concat(monthly_ds_v, dim='time')
    ds_speed = xr.concat(monthly_ds_speed, dim='time')

    # Aggregates
    ds_u_clim = ds_u.groupby('time.month').mean()
    ds_v_clim = ds_v.groupby('time.month').mean()
    ds_speed_clim = ds_speed.groupby('time.month').mean()

    ds_u_seas = ds_u.groupby('time.season').mean()
    ds_v_seas = ds_v.groupby('time.season').mean()
    ds_speed_seas = ds_speed.groupby('time.season').mean()

    ranges = {
        'ts': [float(ds_speed.min()), float(ds_speed.max())],
        'clim': [float(ds_speed_clim.min()), float(ds_speed_clim.max())],
        'seas': [float(ds_speed_seas.min()), float(ds_speed_seas.max())]
    }
    
    # ==========================================
    # 3. FUNGSI PLOT
    # ==========================================
    
    def make_colorbar_img(vmin, vmax, path, label="Wind Speed (m/s)"):
        fig = plt.figure(figsize=(1.5, 6), dpi=150)
        fig.patch.set_alpha(0)
        ax = fig.add_axes([0.2, 0.05, 0.4, 0.9])
        norm = plt.Normalize(vmin=vmin, vmax=vmax)
        cb = plt.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=CMAP_MAP), cax=ax)
        cb.set_label(label, color=TEXT_COLOR, weight='bold', fontsize=24)
        cb.ax.yaxis.set_tick_params(color=TEXT_COLOR)
        plt.setp(cb.ax.get_yticklabels(), color=TEXT_COLOR, weight='bold', fontsize=22)
        cb.outline.set_edgecolor(TEXT_COLOR)
        plt.savefig(path, transparent=True, bbox_inches='tight')
        plt.close(fig)

    def plot_map_full_coverage(speed_da, u_da, v_da, filename, vmin, vmax):
        lat = speed_da.latitude.values
        lon = speed_da.longitude.values
        n_lat, n_lon = int(len(lat)*3), int(len(lon)*3)
        new_lat = np.linspace(lat.min(), lat.max(), n_lat)
        new_lon = np.linspace(lon.min(), lon.max(), n_lon)
        speed_smooth = speed_da.interp(latitude=new_lat, longitude=new_lon, method='linear')
        vals = speed_smooth.values 

        aspect = (lon.max()-lon.min())/(lat.max()-lat.min())
        fig = plt.figure(figsize=(5*aspect, 5), dpi=DPI_MAP, frameon=False)
        ax = plt.axes(projection=ccrs.PlateCarree())
        ax.set_extent([lon.min(), lon.max(), lat.min(), lat.max()], crs=ccrs.PlateCarree())
        
        ax.pcolormesh(new_lon, new_lat, vals, transform=ccrs.PlateCarree(),
                      vmin=vmin, vmax=vmax, cmap=CMAP_MAP, shading='auto', zorder=1)
        ax.add_feature(cfeature.COASTLINE, edgecolor=COASTLINE_COLOR, linewidth=1.5, zorder=2)
        skip = 2 
        ax.quiver(lon[::skip], lat[::skip], u_da.values[::skip, ::skip], v_da.values[::skip, ::skip],
                  transform=ccrs.PlateCarree(), color=ARROW_COLOR, 
                  scale=None, scale_units='xy', alpha=0.9, width=0.005, headwidth=3.5, zorder=3)
        ax.set_axis_off()
        plt.savefig(filename, transparent=True, bbox_inches='tight', pad_inches=0)
        plt.close(fig)

    def plot_wind_rose_magma(u_arr, v_arr, filename):
        u_flat = u_arr.flatten()
        v_flat = v_arr.flatten()
        mask = ~np.isnan(u_flat) & ~np.isnan(v_flat)
        u_clean = u_flat[mask]
        v_clean = v_flat[mask]
        if len(u_clean) == 0: return

        ws = np.sqrt(u_clean**2 + v_clean**2)
        wd = (270 - np.degrees(np.arctan2(v_clean, u_clean))) % 360
        n_sectors = 16
        max_val = np.max(ws) if len(ws) > 0 else 0
        ws_bins = [0, 2, 4, 6, 8, 10, max_val if max_val > 10 else 10.1]
        
        sector_width = 360 / n_sectors
        sector_indices = (np.round(wd / sector_width) % n_sectors).astype(int)
        ws_indices = np.digitize(ws, ws_bins) - 1
        freqs = np.zeros((len(ws_bins)-1, n_sectors))
        for i in range(len(ws)):
            if 0 <= ws_indices[i] < len(ws_bins)-1:
                freqs[ws_indices[i], sector_indices[i]] += 1
        total = len(ws)
        freqs_pct = (freqs / total) * 100 if total > 0 else freqs

        fig = plt.figure(figsize=(6, 6), dpi=DPI_ROSE)
        fig.patch.set_alpha(0) 
        ax = fig.add_subplot(111, polar=True)
        ax.set_facecolor('none') 
        ax.set_theta_zero_location('N')
        ax.set_theta_direction(-1)
        
        colors = cm.magma(np.linspace(0.2, 1, len(ws_bins)-1))
        angles = np.linspace(0, 2 * np.pi, n_sectors, endpoint=False)
        width = (2 * np.pi / n_sectors) * 0.9
        bottom = 0
        labels_legend = [f"{ws_bins[i]}-{ws_bins[i+1]} m/s" for i in range(len(ws_bins)-1)]
        labels_legend[-1] = f"> {ws_bins[-2]} m/s"

        for i in range(len(ws_bins)-1):
            ax.bar(angles, freqs_pct[i, :], width=width, bottom=bottom, color=colors[i], 
                   label=labels_legend[i], alpha=0.9, edgecolor='#1a1a1a', linewidth=0.5)
            bottom += freqs_pct[i, :]
            
        ax.tick_params(axis='x', colors=TEXT_COLOR, labelsize=22, pad=10)
        ax.tick_params(axis='y', colors='gray', labelsize=18)
        ax.yaxis.grid(True, color='gray', linestyle=':', alpha=0.4)
        ax.xaxis.grid(True, color='gray', linestyle=':', alpha=0.4)
        ax.spines['polar'].set_visible(False)
        ax.set_xticklabels(['N','NE','E','SE','S','SW','W','NW'])
        legend = ax.legend(loc='center left', bbox_to_anchor=(1.15, 0.5),   
                           title="Wind Speed (m/s)", frameon=False,
                           fontsize=18, title_fontsize=20, labelspacing=0.8)
        plt.setp(legend.get_title(), color=TEXT_COLOR, fontweight='bold')
        for text in legend.get_texts():
            plt.setp(text, color=TEXT_COLOR)
        plt.savefig(filename, transparent=True, bbox_inches='tight')
        plt.close(fig)

    # ==========================================
    # 4. EKSEKUSI GENERATE AREA FRAMES
    # ==========================================
    labels_ts = []
    print("\n Rendering Time Series (Area)...")
    for i, t in enumerate(ds_speed.time):
        t_str = pd.to_datetime(t.values).strftime('%Y-%m')
        labels_ts.append(t_str)
        fname_map = f"{dirs['map_ts']}/map_{i}.png"
        plot_map_full_coverage(ds_speed.isel(time=i), ds_u.isel(time=i), ds_v.isel(time=i), fname_map, ranges['ts'][0], ranges['ts'][1])
        fname_rose = f"{dirs['rose_area_ts']}/rose_{i}.png"
        plot_wind_rose_magma(ds_u.isel(time=i).values, ds_v.isel(time=i).values, fname_rose)
        if i % 50 == 0: print(f"   Frame TS {i}...", end='\r')

    print("\n Rendering Climatology (Area)...")
    for i in range(12):
        fname_map = f"{dirs['map_clim']}/map_{i}.png"
        plot_map_full_coverage(ds_speed_clim.sel(month=i+1), ds_u_clim.sel(month=i+1), ds_v_clim.sel(month=i+1), fname_map, ranges['clim'][0], ranges['clim'][1])
        fname_rose = f"{dirs['rose_area_clim']}/rose_{i}.png"
        plot_wind_rose_magma(ds_u_clim.sel(month=i+1).values, ds_v_clim.sel(month=i+1).values, fname_rose)

    print("\n Rendering Seasonal (Area)...")
    season_order = ['DJF', 'MAM', 'JJA', 'SON']
    for i, s in enumerate(season_order):
        try:
            fname_map = f"{dirs['map_seas']}/map_{i}.png"
            plot_map_full_coverage(ds_speed_seas.sel(season=s), ds_u_seas.sel(season=s), ds_v_seas.sel(season=s), fname_map, ranges['seas'][0], ranges['seas'][1])
            fname_rose = f"{dirs['rose_area_seas']}/rose_{i}.png"
            plot_wind_rose_magma(ds_u_seas.sel(season=s).values, ds_v_seas.sel(season=s).values, fname_rose)
        except: pass

    # ==========================================
    # 5. GRID ROSES (DYNAMIC PER MODE)
    # ==========================================
    print("\n Rendering Dynamic Grid Roses (TS Static, Clim 12, Seas 4)...")
    
    # Smoothing 3x3 untuk representasi grid
    ds_u_smooth = ds_u.rolling(latitude=3, longitude=3, center=True, min_periods=1).mean()
    ds_v_smooth = ds_v.rolling(latitude=3, longitude=3, center=True, min_periods=1).mean()
    
    # Siapkan data smooth untuk Clim dan Seas juga
    ds_u_clim_smooth = ds_u_clim.rolling(latitude=3, longitude=3, center=True, min_periods=1).mean()
    ds_v_clim_smooth = ds_v_clim.rolling(latitude=3, longitude=3, center=True, min_periods=1).mean()
    ds_u_seas_smooth = ds_u_seas.rolling(latitude=3, longitude=3, center=True, min_periods=1).mean()
    ds_v_seas_smooth = ds_v_seas.rolling(latitude=3, longitude=3, center=True, min_periods=1).mean()

    lats = ds_speed.latitude.values
    lons = ds_speed.longitude.values
    
    count = 0
    total_grid = len(lats) * len(lons)
    
    for i, lat in enumerate(lats):
        for j, lon in enumerate(lons):
            # Cek NaN pada data TS (cukup cek index 0)
            if np.isnan(ds_u_smooth.values[0, i, j]): continue
            
            # --- 1. TS MODE: STATIC AVERAGE (1979-2024) ---
            # Sesuai request: "Time Series di rata ratakan saja"
            fname_ts = f"{dirs['rose_grid_ts']}/rose_{i}_{j}.png"
            plot_wind_rose_magma(ds_u_smooth.values[:, i, j], ds_v_smooth.values[:, i, j], fname_ts)
            
            # --- 2. CLIM MODE: DYNAMIC (12 Months) ---
            # Sesuai request: "Climatology rata rata climatology saja" (Kita buat per bulan agar tidak ambigu/sama dengan TS)
            for m in range(12):
                fname_clim = f"{dirs['rose_grid_clim']}/rose_{i}_{j}_{m}.png"
                # Ambil data bulan ke-m untuk grid i,j
                u_val = ds_u_clim_smooth.values[m, i, j]
                v_val = ds_v_clim_smooth.values[m, i, j]
                # Wind rose butuh array, walau cuma 1 nilai (rata-rata bulan itu), 
                # Tapi wind rose 1 nilai itu jelek. 
                # *Koreksi:* "Rata-rata climatology" berarti kita ambil SEMUA Januari dari 1979-2024 untuk grid itu.
                # Data 'ds_u_smooth' memiliki dimensi (Time, Lat, Lon).
                # Kita harus memfilter time berdasarkan bulan 'm+1'.
                
                # Mengambil slice waktu untuk bulan tertentu dari data original (yg sudah di smooth spasial)
                u_month_slice = ds_u_smooth.sel(time=ds_u_smooth.time.dt.month == (m+1)).values[:, i, j]
                v_month_slice = ds_v_smooth.sel(time=ds_v_smooth.time.dt.month == (m+1)).values[:, i, j]
                
                plot_wind_rose_magma(u_month_slice, v_month_slice, fname_clim)

            # --- 3. SEAS MODE: DYNAMIC (4 Seasons) ---
            for s_idx, s_name in enumerate(season_order):
                try:
                    fname_seas = f"{dirs['rose_grid_seas']}/rose_{i}_{j}_{s_idx}.png"
                    # Ambil slice waktu untuk season tertentu
                    u_seas_slice = ds_u_smooth.sel(time=ds_u_smooth.time.dt.season == s_name).values[:, i, j]
                    v_seas_slice = ds_v_smooth.sel(time=ds_v_smooth.time.dt.season == s_name).values[:, i, j]
                    plot_wind_rose_magma(u_seas_slice, v_seas_slice, fname_seas)
                except: pass
            
            count += 1
            if count % 20 == 0: print(f"   Processing Grid {count}/{total_grid}...", end='\r')

    # ==========================================
    # 6. EXPORT JSON
    # ==========================================
    print("\nMaking Final JSON...")
    
    make_colorbar_img(ranges['ts'][0], ranges['ts'][1], "colorbar_wind_ts.png", "Wind Speed (m/s) [Time Series]")
    make_colorbar_img(ranges['clim'][0], ranges['clim'][1], "colorbar_wind_clim.png", "Wind Speed (m/s) [Monthly Clim]")
    make_colorbar_img(ranges['seas'][0], ranges['seas'][1], "colorbar_wind_seas.png", "Wind Speed (m/s) [Seasonal]")

    aa_speed_ts = ds_speed.mean(dim=['latitude', 'longitude'])
    aa_speed_clim = ds_speed_clim.mean(dim=['latitude', 'longitude'])
    aa_speed_seas = ds_speed_seas.mean(dim=['latitude', 'longitude'])
    
    def to_list(da):
        return [round(float(x), 2) if not np.isnan(x) else None for x in da.values]

    output_data = {
        "metadata": {
            "latitudes": lats.tolist(),
            "longitudes": lons.tolist(),
            "labels_ts": labels_ts,
            "labels_clim": ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"],
            "labels_seas": season_order,
            "paths": {
                "map": {
                    "ts": f"{dirs['map_ts']}/map_",
                    "clim": f"{dirs['map_clim']}/map_",
                    "seas": f"{dirs['map_seas']}/map_"
                },
                "rose": {
                    "area_ts": f"{dirs['rose_area_ts']}/rose_",
                    "area_clim": f"{dirs['rose_area_clim']}/rose_",
                    "area_seas": f"{dirs['rose_area_seas']}/rose_",
                    # [MODIFIKASI] Path baru untuk Grid
                    "grid_ts": f"{dirs['rose_grid_ts']}/rose_",
                    "grid_clim": f"{dirs['rose_grid_clim']}/rose_",
                    "grid_seas": f"{dirs['rose_grid_seas']}/rose_"
                }
            },
            "colorbar": {
                "ts": "colorbar_wind_ts.png",
                "clim": "colorbar_wind_clim.png",
                "seas": "colorbar_wind_seas.png"
            }
        },
        "area_average": {
            "ts": to_list(aa_speed_ts),
            "clim": to_list(aa_speed_clim),
            "seas": [round(float(aa_speed_seas.sel(season=s)), 2) for s in season_order]
        },
        "grid_data": {}
    }

    speed_smooth = ds_speed.rolling(latitude=3, longitude=3, center=True, min_periods=1).mean()
    clim_smooth = ds_speed_clim.rolling(latitude=3, longitude=3, center=True, min_periods=1).mean()
    seas_smooth = ds_speed_seas.rolling(latitude=3, longitude=3, center=True, min_periods=1).mean()

    for i, lat in enumerate(lats):
        for j, lon in enumerate(lons):
            grid_key = f"{i},{j}"
            ts_vals = to_list(speed_smooth[:, i, j])
            if all(x is None for x in ts_vals): continue
            
            output_data["grid_data"][grid_key] = {
                "ts": ts_vals,
                "clim": to_list(clim_smooth[:, i, j]),
                "seas": [round(float(seas_smooth.sel(season=s)[i, j]), 2) if s in seas_smooth.season else None for s in season_order]
            }

    with open('wind_data_images.json', 'w') as f:
        json.dump(output_data, f)

    print("\nSELESAI! Data Grid Wind Rose sudah digenerate (Static TS, Dynamic Clim/Seas).")

if __name__ == "__main__":
    process_wind_images_final_v3()

GENERATOR V4: GRID WIND ROSES (DYNAMIC CLIM/SEAS)
Membaca Data Angin (U & V) 1979-2024...
   [2024] OK
Menggabungkan Dataset...

 Rendering Time Series (Area)...
   Frame TS 550...
 Rendering Climatology (Area)...

 Rendering Seasonal (Area)...

 Rendering Dynamic Grid Roses (TS Static, Clim 12, Seas 4)...
   Processing Grid 280/289...
Making Final JSON...

SELESAI! Data Grid Wind Rose sudah digenerate (Static TS, Dynamic Clim/Seas).
